# Delta Hedging Concept

This notebook works through a manual delta-hedging example. The goal is to show why an option position has stock exposure, how a stock hedge offsets that exposure locally, and why the hedge must be adjusted when Delta changes.

## Core idea

Hedging means taking another position to offset a risk exposure. An option has stock exposure because its value changes when the underlying stock price changes.

Delta is the local estimate of that stock exposure. If a call has Delta `0.60`, then one long call behaves approximately like `0.60` shares of stock for a small move in the underlying.

For a long call:

\[
\text{target stock hedge} = -\Delta
\]

So a long call with Delta `0.60` is hedged by shorting `0.60` shares.

For a short call:

\[
\text{target stock hedge} = +\Delta
\]

So a short call with Delta `0.60` is hedged by buying `0.60` shares.

## Why the hedge changes

Delta is not fixed. It changes when the stock moves, when time passes, and when volatility changes. Gamma measures how sensitive Delta is to the stock price.

If Gamma is high, Delta can change quickly. That means the hedge has to be adjusted more often.

Continuous hedging assumes the hedge can be adjusted all the time with no cost. Real hedging is discrete. A trader might rebalance every hour, every day, or after a large move. Because the hedge is not updated continuously, the position develops hedging error.

In [ ]:
from pathlib import Path

import pandas as pd

## Manual example setup

Assume:

- Long 1 call option
- Initial stock price: `100`
- Initial call value: `8.00`
- Initial Delta: `0.60`
- Hedge: short `0.60` shares
- Transaction cost: `5 bps`, or `0.05%`, of traded stock notional

The option prices and Deltas below are manual scenario values. They are not being generated from a pricing model in this notebook.

In [ ]:
transaction_cost_bps = 5
transaction_cost_rate = transaction_cost_bps / 10_000

scenario = pd.DataFrame(
    [
        {
            "step": 0,
            "description": "Initial hedge",
            "stock_price": 100.00,
            "call_value": 8.00,
            "delta": 0.60,
        },
        {
            "step": 1,
            "description": "Stock rises",
            "stock_price": 102.00,
            "call_value": 9.30,
            "delta": 0.68,
        },
        {
            "step": 2,
            "description": "Stock falls",
            "stock_price": 98.00,
            "call_value": 6.10,
            "delta": 0.42,
        },
        {
            "step": 3,
            "description": "Stock recovers",
            "stock_price": 101.00,
            "call_value": 8.10,
            "delta": 0.57,
        },
    ]
)

scenario

## Build the hedge table

The target stock position for a long call is negative Delta. If Delta rises from `0.60` to `0.68`, the hedge must move from short `0.60` shares to short `0.68` shares. That means shorting an additional `0.08` shares.

In [ ]:
def build_manual_hedge_table(scenario: pd.DataFrame, transaction_cost_rate: float) -> pd.DataFrame:
    """Build a manual delta-hedging table for a long one-call position."""
    rows = []

    previous_stock_price = None
    previous_call_value = None
    previous_stock_position = None
    cumulative_gross_pnl = 0.0
    cumulative_transaction_cost = 0.0
    cumulative_net_pnl = 0.0

    for _, row in scenario.iterrows():
        target_stock_position = -row["delta"]

        if previous_stock_position is None:
            stock_trade = target_stock_position
            stock_pnl = 0.0
            option_pnl = 0.0
            gross_pnl = 0.0
        else:
            stock_pnl = previous_stock_position * (row["stock_price"] - previous_stock_price)
            option_pnl = row["call_value"] - previous_call_value
            gross_pnl = stock_pnl + option_pnl
            stock_trade = target_stock_position - previous_stock_position

        trade_notional = abs(stock_trade) * row["stock_price"]
        transaction_cost = trade_notional * transaction_cost_rate
        net_pnl = gross_pnl - transaction_cost

        cumulative_gross_pnl += gross_pnl
        cumulative_transaction_cost += transaction_cost
        cumulative_net_pnl += net_pnl

        rows.append(
            {
                "step": row["step"],
                "description": row["description"],
                "stock_price": row["stock_price"],
                "call_value": row["call_value"],
                "delta": row["delta"],
                "target_stock_position": target_stock_position,
                "stock_trade_to_rebalance": stock_trade,
                "stock_pnl_since_last_step": stock_pnl,
                "option_pnl_since_last_step": option_pnl,
                "gross_pnl_since_last_step": gross_pnl,
                "transaction_cost": transaction_cost,
                "net_pnl_since_last_step": net_pnl,
                "cumulative_gross_pnl": cumulative_gross_pnl,
                "cumulative_transaction_cost": cumulative_transaction_cost,
                "cumulative_net_pnl": cumulative_net_pnl,
            }
        )

        previous_stock_price = row["stock_price"]
        previous_call_value = row["call_value"]
        previous_stock_position = target_stock_position

    return pd.DataFrame(rows)


hedge_table = build_manual_hedge_table(scenario, transaction_cost_rate)
hedge_table.round(4)

## Reading the table

At the start, the call has Delta `0.60`, so the hedge is short `0.60` shares.

When the stock rises from `100` to `102`, the call gains value, but the short stock hedge loses money:

\[
\text{stock hedge P\&L} = -0.60 \times (102 - 100) = -1.20
\]

The call value rises from `8.00` to `9.30`, so the option gains `1.30`.

The combined gross P&L is:

\[
1.30 - 1.20 = 0.10
\]

The hedge reduced the stock exposure, but it did not remove all risk. Delta changed from `0.60` to `0.68`, so the hedge must be adjusted.

In [ ]:
hedge_table[
    [
        "step",
        "description",
        "stock_price",
        "delta",
        "target_stock_position",
        "stock_trade_to_rebalance",
        "stock_pnl_since_last_step",
        "option_pnl_since_last_step",
        "gross_pnl_since_last_step",
        "transaction_cost",
        "net_pnl_since_last_step",
        "cumulative_net_pnl",
    ]
].round(4)

## Delta-neutrality

A position is delta-neutral when the option Delta and stock hedge roughly offset each other.

For the initial hedge:

\[
\text{net Delta} = 0.60 - 0.60 = 0
\]

That means the combined position is locally protected against a very small stock move.

But after the stock moves, the call Delta changes. If Delta becomes `0.68` and the hedge is still only short `0.60` shares, the position is no longer delta-neutral:

\[
\text{net Delta} = 0.68 - 0.60 = 0.08
\]

The position still has positive stock exposure until the hedge is rebalanced.

In [ ]:
delta_neutrality = hedge_table[
    [
        "step",
        "description",
        "delta",
        "target_stock_position",
        "stock_trade_to_rebalance",
    ]
].copy()

delta_neutrality["net_delta_after_rebalance"] = (
    delta_neutrality["delta"] + delta_neutrality["target_stock_position"]
)

delta_neutrality.round(4)

## Transaction-cost example

Rebalancing is not free. If the hedge must be adjusted often, transaction costs accumulate. The cost used here is:

\[
\text{transaction cost} = |\text{shares traded}| \times \text{stock price} \times \text{cost rate}
\]

With a 5 bps transaction cost, shorting or buying a small fraction of a share creates a small cost. In a real book with many contracts, these costs can become material.

In [ ]:
transaction_cost_table = hedge_table[
    [
        "step",
        "description",
        "stock_trade_to_rebalance",
        "stock_price",
        "transaction_cost",
        "cumulative_transaction_cost",
    ]
].copy()

transaction_cost_table.round(6)

## Cost sensitivity

The same hedge path can look different under different cost assumptions. This is why transaction costs matter in a discrete hedging project.

In [ ]:
cost_scenarios = []

for bps in [0, 1, 5, 10, 25]:
    rate = bps / 10_000
    table = build_manual_hedge_table(scenario, rate)
    cost_scenarios.append(
        {
            "transaction_cost_bps": bps,
            "total_gross_pnl": table["cumulative_gross_pnl"].iloc[-1],
            "total_transaction_cost": table["cumulative_transaction_cost"].iloc[-1],
            "total_net_pnl": table["cumulative_net_pnl"].iloc[-1],
        }
    )

cost_sensitivity = pd.DataFrame(cost_scenarios)
cost_sensitivity.round(6)

## Save outputs

In [ ]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

hedge_table_path = output_dir / "manual_delta_hedge_table.csv"
cost_sensitivity_path = output_dir / "manual_delta_hedge_transaction_cost_sensitivity.csv"

hedge_table.to_csv(hedge_table_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)

hedge_table_path, cost_sensitivity_path

## Takeaway

Delta hedging offsets stock exposure locally, but it is not a perfect lock. Delta changes as the stock moves because of Gamma. Real hedging happens at discrete times, not continuously, so hedging error appears between rebalances. Transaction costs make frequent rebalancing expensive.

This is why the simulation later in the project measures hedging error instead of assuming the hedge works perfectly.